# Fed NLP Pipeline — Runner

This notebook controls the full data collection and processing pipeline.

| Step | Script | Description |
|------|--------|-------------|
| 1 | `fed_scraper.py` | Download raw HTML/PDF from Fed website |
| 2 | `convert_to_txt.py` | Convert raw files to plain TXT |
| 3 | `build_metadata.py` | Build metadata CSV |
| — | `weekly_update.py` | Incremental update (steps 1–3 for new docs only) |

> **Run cells in order on first use. For subsequent updates, jump to Section 4.**

---
## 0. Setup

In [41]:
import subprocess
import sys
import os
from pathlib import Path

# ── Set working directory ─────────────────────────────────────────────────
PIPELINE_ROOT = Path("/Users/zifeimeng/Desktop/FED Github/scripts").resolve()
os.chdir(PIPELINE_ROOT)
print(f"Working directory: {PIPELINE_ROOT}")

# ── Helper: run a .py script and stream output live to notebook ──────────
def run(script: str, *args: str, check: bool = True) -> int:
    cmd = [sys.executable, str(PIPELINE_ROOT / script)] + list(args)
    print(f"\n{'='*60}")
    print(f"Running: {' '.join(cmd)}")
    print(f"{'='*60}\n")

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()

    if check and process.returncode != 0:
        raise RuntimeError(f"Script failed with return code {process.returncode}")

    print(f"\n✓ Finished with return code {process.returncode}")
    return process.returncode

Working directory: /Users/zifeimeng/Desktop/FED Github/scripts


In [6]:
# ── Install dependencies (only needs to be run once) ─────────────────────
!pip install selenium webdriver-manager requests beautifulsoup4 lxml pdfplumber pandas -q

In [47]:
import shutil, json

# 清空raw文件夹
for doc_type in ["statements", "minutes", "press_conf"]:
    shutil.rmtree(PIPELINE_ROOT / "data" / "raw" / doc_type, ignore_errors=True)
    (PIPELINE_ROOT / "data" / "raw" / doc_type).mkdir(parents=True)

# 清空checkpoint
with open(PIPELINE_ROOT / "data" / ".checkpoints" / "scraper.json", "w") as f:
    json.dump({}, f)

print("Done — raw files and checkpoint cleared.")

Done — raw files and checkpoint cleared.


---
## 1. Scrape Raw Documents

Downloads HTML/PDF files from `federalreserve.gov/monetarypolicy/materials/`.

- **First run**: scrapes all pages (2000–present), takes ~20–30 min
- Already-downloaded docs are skipped automatically (checkpoint-based)
- Output: `data/raw/{statements, minutes, press_conf}/`

In [48]:
run(
    "fed_scraper.py",
    "--types", "statements", "minutes", "press_conf",
    "--start-year", "2000",
    "--end-year", "2026",
)


Running: /opt/anaconda3/bin/python /Users/zifeimeng/Desktop/FED Github/scripts/fed_scraper.py --types statements minutes press_conf --start-year 2000 --end-year 2026

2026-04-20 14:33:09,466 [INFO] fed_scraper — Target types  : ['statements', 'minutes', 'press_conf']
2026-04-20 14:33:09,466 [INFO] fed_scraper — Date range    : 2000–2026
2026-04-20 14:33:09,466 [INFO] fed_scraper — In checkpoint : 0 documents
2026-04-20 14:33:09,466 [INFO] fed_scraper — Fetching current FOMC calendar …
2026-04-20 14:33:10,165 [INFO] fed_scraper —   89 dates found.
2026-04-20 14:33:16,946 [INFO] fed_scraper — Total unique meeting dates collected: 89
2026-04-20 14:33:16,946 [INFO] fed_scraper —   Range: 20120125 → 20260318
2026-04-20 14:33:18,361 [INFO] fed_scraper —   ✓ stmt_20120125 (html)
2026-04-20 14:33:20,089 [INFO] fed_scraper —   ✓ min_20120125 (pdf)
2026-04-20 14:33:21,487 [INFO] fed_scraper —   ✓ pc_20120125 (html)
2026-04-20 14:33:22,879 [INFO] fed_scraper —   ✓ stmt_20120425 (html)
2026-04-20

0

In [50]:
run(
    "fed_scraper.py",
    "--types", "statements", "minutes", "press_conf",
    "--start-year", "2000",
    "--end-year", "2011",
)


Running: /opt/anaconda3/bin/python /Users/zifeimeng/Desktop/FED Github/scripts/fed_scraper.py --types statements minutes press_conf --start-year 2000 --end-year 2011

2026-04-20 14:48:46,755 [INFO] fed_scraper — Target types  : ['statements', 'minutes', 'press_conf']
2026-04-20 14:48:46,756 [INFO] fed_scraper — Date range    : 2000–2011
2026-04-20 14:48:46,756 [INFO] fed_scraper — In checkpoint : 263 documents
2026-04-20 14:48:46,756 [INFO] fed_scraper — Loaded 101 hardcoded dates (2000–2011).
2026-04-20 14:48:46,756 [INFO] fed_scraper — Total unique meeting dates: 101
2026-04-20 14:48:46,756 [INFO] fed_scraper —   Range: 20000202 → 20111213
2026-04-20 14:49:39,117 [INFO] fed_scraper —   ✓ stmt_20060131 (html)
2026-04-20 14:49:40,892 [INFO] fed_scraper —   ✓ stmt_20060328 (html)
2026-04-20 14:49:42,786 [INFO] fed_scraper —   ✓ stmt_20060510 (html)
2026-04-20 14:49:44,492 [INFO] fed_scraper —   ✓ stmt_20060629 (html)
2026-04-20 14:49:46,285 [INFO] fed_scraper —   ✓ stmt_20060808 (html)

0

In [51]:
run(
    "fed_scraper.py",
    "--types", "statements", "minutes", "press_conf",
    "--start-year", "2000",
    "--end-year", "2011",
)


Running: /opt/anaconda3/bin/python /Users/zifeimeng/Desktop/FED Github/scripts/fed_scraper.py --types statements minutes press_conf --start-year 2000 --end-year 2011

2026-04-20 15:04:34,330 [INFO] fed_scraper — Target types  : ['statements', 'minutes', 'press_conf']
2026-04-20 15:04:34,331 [INFO] fed_scraper — Date range    : 2000–2011
2026-04-20 15:04:34,331 [INFO] fed_scraper — In checkpoint : 348 documents
2026-04-20 15:04:34,331 [INFO] fed_scraper — Loaded 101 hardcoded dates (2000–2011).
2026-04-20 15:04:34,331 [INFO] fed_scraper — Total unique meeting dates: 101
2026-04-20 15:04:34,331 [INFO] fed_scraper —   Range: 20000202 → 20111213
2026-04-20 15:04:36,274 [INFO] fed_scraper —   ✓ stmt_20000202 (html)
2026-04-20 15:04:38,197 [INFO] fed_scraper —   ✓ min_20000202 (html)
2026-04-20 15:04:40,345 [INFO] fed_scraper —   ✓ stmt_20000321 (html)
2026-04-20 15:04:42,146 [INFO] fed_scraper —   ✓ min_20000321 (html)
2026-04-20 15:04:43,971 [INFO] fed_scraper —   ✓ stmt_20000516 (html)
2

0

In [52]:
import json
with open(PIPELINE_ROOT / "data/.checkpoints/scraper.json") as f:
    ckpt = json.load(f)

by_year = {}
for doc_id, info in ckpt.items():
    if info.get("status") == "ok":
        year = info["date"][:4]
        by_year[year] = by_year.get(year, 0) + 1

for year in sorted(by_year):
    print(f"{year}: {by_year[year]} documents")

2000: 16 documents
2001: 19 documents
2002: 10 documents
2003: 8 documents
2004: 8 documents
2005: 8 documents
2006: 16 documents
2007: 15 documents
2008: 17 documents
2009: 16 documents
2010: 16 documents
2011: 19 documents
2012: 15 documents
2013: 12 documents
2014: 12 documents
2015: 12 documents
2016: 12 documents
2017: 12 documents
2018: 12 documents
2019: 24 documents
2020: 26 documents
2021: 24 documents
2022: 24 documents
2023: 24 documents
2024: 24 documents
2025: 25 documents
2026: 5 documents


In [53]:
import requests
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}
BASE_URL = "https://www.federalreserve.gov"

# 直接测试几个2003年Minutes URL
test_urls = [
    "/fomc/minutes/20030812.htm",
    "/fomc/minutes/20030506.htm",
    "/fomc/minutes/20030129.htm",
]
for path in test_urls:
    resp = requests.get(BASE_URL + path, headers=HEADERS, timeout=30)
    print(f"{resp.status_code}: {path}")

200: /fomc/minutes/20030812.htm
200: /fomc/minutes/20030506.htm
200: /fomc/minutes/20030129.htm


In [54]:
# ── Check what was downloaded ─────────────────────────────────────────────
from pathlib import Path

for doc_type in ["statements", "minutes", "press_conf"]:
    path = PIPELINE_ROOT / "data" / "raw" / doc_type
    files = list(path.glob("*")) if path.exists() else []
    print(f"{doc_type:15s}: {len(files):4d} files")

statements     :  158 files
minutes        :  183 files
press_conf     :   90 files


---
## 2. Convert Raw Files to TXT

- PDF → `pdfplumber`
- HTML → `BeautifulSoup` (strips tags, preserves paragraph breaks)
- Output: `data/processed/{statements, minutes, press_conf}/`

In [55]:
run("convert_to_txt.py")


Running: /opt/anaconda3/bin/python /Users/zifeimeng/Desktop/FED Github/scripts/convert_to_txt.py

2026-04-20 15:17:46,349 [INFO] convert_to_txt — 
[statements] 158 raw files found.
2026-04-20 15:17:46,351 [INFO] convert_to_txt —   ✓ stmt_20000202
2026-04-20 15:17:46,353 [INFO] convert_to_txt —   ✓ stmt_20000321
2026-04-20 15:17:46,355 [INFO] convert_to_txt —   ✓ stmt_20000516
2026-04-20 15:17:46,356 [INFO] convert_to_txt —   ✓ stmt_20000628
2026-04-20 15:17:46,358 [INFO] convert_to_txt —   ✓ stmt_20000822
2026-04-20 15:17:46,360 [INFO] convert_to_txt —   ✓ stmt_20001003
2026-04-20 15:17:46,361 [INFO] convert_to_txt —   ✓ stmt_20001115
2026-04-20 15:17:46,363 [INFO] convert_to_txt —   ✓ stmt_20001219
2026-04-20 15:17:46,364 [INFO] convert_to_txt —   ✓ stmt_20010103
2026-04-20 15:17:46,366 [INFO] convert_to_txt —   ✓ stmt_20010131
2026-04-20 15:17:46,367 [INFO] convert_to_txt —   ✓ stmt_20010320
2026-04-20 15:17:46,369 [INFO] convert_to_txt —   ✓ stmt_20010418
2026-04-20 15:17:46,370 [I

0

In [56]:
# ── Spot-check: print first 500 chars of a converted file ────────────────
from pathlib import Path

sample_files = sorted((PIPELINE_ROOT / "data/processed/statements").glob("*.txt"))
if sample_files:
    sample = sample_files[-1]   # most recent
    print(f"Sample: {sample.name}\n")
    print(sample.read_text()[:500])
else:
    print("No TXT files found yet.")

Sample: stmt_20260318.txt

ï»¿ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Federal Reserve Board - Federal Reserve issues FOMC statement 
 
 
 
 
 
 
 
 
 
 
 
 
 Skip to main content 
 
 
 
 
 
 
 An official website of the United States Government 
 
 Here's how you know 
 
 
 
 
 Official websites use .gov 
 
 A .gov website belongs to an official government organization in the United States. 
 
 
 
 Secure .gov websites use HTTPS 
 
 A lock ( Lock Locked padlock icon ) or https:// means you've safely connected to the .gov w


---
## 3. Build Metadata CSV

Scans all processed TXT files and produces `data/metadata.csv` with columns:
`doc_id`, `doc_type`, `doc_date`, `meeting_date`, `download_date`, `year`, `source`, `file_name`, `file_ext`, `speaker`, `chair_regime`, `local_path`

In [70]:
run("build_metadata.py")


Running: /opt/anaconda3/bin/python /Users/zifeimeng/Desktop/FED Github/scripts/build_metadata.py

2026-04-20 16:00:31,709 [INFO] build_metadata — Loaded 431 checkpoint entries.
2026-04-20 16:00:31,710 [INFO] build_metadata — [statements] 158 processed TXT files.
2026-04-20 16:00:31,719 [INFO] build_metadata — [minutes] 183 processed TXT files.
2026-04-20 16:00:31,727 [INFO] build_metadata — [press_conf] 90 processed TXT files.
2026-04-20 16:00:31,741 [INFO] build_metadata — 
Metadata saved → /Users/zifeimeng/Desktop/FED Github/scripts/data/metadata.csv
2026-04-20 16:00:31,741 [INFO] build_metadata — Total documents: 431
2026-04-20 16:00:31,744 [INFO] build_metadata — 
            count  year_min  year_max
doc_type                             
minutes       183      2000      2026
press_conf     90      2011      2026
statements    158      2000      2026

✓ Finished with return code 0


0

In [71]:
# ── Preview metadata ──────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv(PIPELINE_ROOT / "data/metadata.csv")
print(f"Total documents: {len(df)}")
print()
print(df.groupby("doc_type").agg(
    count=("doc_id", "count"),
    year_min=("year", "min"),
    year_max=("year", "max"),
).to_string())
print()
df.head(10)

Total documents: 431

            count  year_min  year_max
doc_type                             
minutes       183      2000      2026
press_conf     90      2011      2026
statements    158      2000      2026



,doc_id,doc_type,doc_date,meeting_date,download_date,year,source,file_name,file_ext,speaker,chair_regime,local_path
0,min_20000202,minutes,2000-02-02,2000-02-02,2026-04-20,2000,https://www.federalreserve.gov/fomc/minutes/20...,min_20000202.html,html,NaN,Greenspan,data/processed/minutes/min_20000202.txt
1,stmt_20000202,statements,2000-02-02,2000-02-02,2026-04-20,2000,https://www.federalreserve.gov/boarddocs/press...,stmt_20000202.html,html,NaN,Greenspan,data/processed/statements/stmt_20000202.txt
2,min_20000321,minutes,2000-03-21,2000-03-21,2026-04-20,2000,https://www.federalreserve.gov/fomc/minutes/20...,min_20000321.html,html,NaN,Greenspan,data/processed/minutes/min_20000321.txt
3,stmt_20000321,statements,2000-03-21,2000-03-21,2026-04-20,2000,https://www.federalreserve.gov/boarddocs/press...,stmt_20000321.html,html,NaN,Greenspan,data/processed/statements/stmt_20000321.txt
4,min_20000516,minutes,2000-05-16,2000-05-16,2026-04-20,2000,https://www.federalreserve.gov/fomc/minutes/20...,min_20000516.html,html,NaN,Greenspan,data/processed/minutes/min_20000516.txt
5,stmt_20000516,statements,2000-05-16,2000-05-16,2026-04-20,2000,https://www.federalreserve.gov/boarddocs/press...,stmt_20000516.html,html,NaN,Greenspan,data/processed/statements/stmt_20000516.txt
6,min_20000628,minutes,2000-06-28,2000-06-28,2026-04-20,2000,https://www.federalreserve.gov/fomc/minutes/20...,min_20000628.html,html,NaN,Greenspan,data/processed/minutes/min_20000628.txt
7,stmt_20000628,statements,2000-06-28,2000-06-28,2026-04-20,2000,https://www.federalreserve.gov/boarddocs/press...,stmt_20000628.html,html,NaN,Greenspan,data/processed/statements/stmt_20000628.txt
8,min_20000822,minutes,2000-08-22,2000-08-22,2026-04-20,2000,https://www.federalreserve.gov/fomc/minutes/20...,min_20000822.html,html,NaN,Greenspan,data/processed/minutes/min_20000822.txt
9,stmt_20000822,statements,2000-08-22,2000-08-22,2026-04-20,2000,https://www.federalreserve.gov/boarddocs/press...,stmt_20000822.html,html,NaN,Greenspan,data/processed/statements/stmt_20000822.txt


---
## 4. Weekly Incremental Update

Run this after each FOMC meeting to pull only NEW documents.
Only scrapes the first 3 pages — already-downloaded docs are skipped automatically.

**FOMC schedule reminder:**
- Statement + Press Conference: released same day as meeting
- Minutes: released ~3 weeks after meeting (usually on a Wednesday)

In [69]:
run("weekly_update.py")


Running: /opt/anaconda3/bin/python /Users/zifeimeng/Desktop/FED Github/scripts/weekly_update.py

2026-04-20 15:59:27,380 [INFO] weekly_update — ============================================================
2026-04-20 15:59:27,381 [INFO] weekly_update — Starting weekly FOMC materials update …
2026-04-20 15:59:27,381 [INFO] weekly_update — Checking first 3 pages (covers ~6 months of meetings)
2026-04-20 15:59:27,620 [INFO] fed_scraper — Target types  : ['statements', 'minutes', 'press_conf']
2026-04-20 15:59:27,621 [INFO] fed_scraper — Date range    : 2000–2026
2026-04-20 15:59:27,621 [INFO] fed_scraper — In checkpoint : 431 documents
2026-04-20 15:59:27,621 [INFO] fed_scraper — Loaded 101 hardcoded dates (2000–2011).
2026-04-20 15:59:27,621 [INFO] fed_scraper — Fetching current FOMC calendar …
2026-04-20 15:59:27,874 [INFO] fed_scraper —   89 new dates from calendar page.
2026-04-20 15:59:27,875 [INFO] fed_scraper — Total unique meeting dates: 190
2026-04-20 15:59:27,875 [INFO] fed_scra

0

In [63]:
# ── Check checkpoint for latest entries ───────────────────────────────────
import json

ckpt_path = PIPELINE_ROOT / "data/.checkpoints/scraper.json"
with open(ckpt_path) as f:
    ckpt = json.load(f)

recent = sorted(ckpt.items(), key=lambda x: x[1].get("date", ""), reverse=True)[:10]
print(f"Total in checkpoint: {len(ckpt)}\n")
print(f"{'doc_id':<20} {'type':<15} {'date':<12} {'status'}")
print("-" * 60)
for doc_id, info in recent:
    print(f"{doc_id:<20} {info.get('type',''):<15} {info.get('date',''):<12} {info.get('status','')}")

Total in checkpoint: 431

doc_id               type            date         status
------------------------------------------------------------
stmt_20260318        statements      20260318     ok
min_20260318         minutes         20260318     ok
pc_20260318          press_conf      20260318     ok
stmt_20260128        statements      20260128     ok
min_20260128         minutes         20260128     ok
stmt_20251210        statements      20251210     ok
min_20251210         minutes         20251210     ok
pc_20251210          press_conf      20251210     ok
stmt_20251029        statements      20251029     ok
min_20251029         minutes         20251029     ok


---
## 5. Quick Diagnostics

In [64]:
# ── File counts across all stages ─────────────────────────────────────────
import pandas as pd
from pathlib import Path

rows = []
for doc_type in ["statements", "minutes", "press_conf"]:
    raw_path  = PIPELINE_ROOT / f"data/raw/{doc_type}"
    proc_path = PIPELINE_ROOT / f"data/processed/{doc_type}"
    raw       = len(list(raw_path.glob("*")))       if raw_path.exists()  else 0
    processed = len(list(proc_path.glob("*.txt")))  if proc_path.exists() else 0
    rows.append({"doc_type": doc_type, "raw": raw, "processed (txt)": processed})

print(pd.DataFrame(rows).to_string(index=False))

  doc_type  raw  processed (txt)
statements  158              158
   minutes  183              183
press_conf   90               90


In [65]:
# ── Failed downloads ──────────────────────────────────────────────────────
import json

with open(PIPELINE_ROOT / "data/.checkpoints/scraper.json") as f:
    ckpt = json.load(f)

failures = {k: v for k, v in ckpt.items() if not str(v.get("status", "")).startswith("ok")}
if failures:
    print(f"Failed downloads ({len(failures)}):")
    for doc_id, info in failures.items():
        print(f"  {doc_id}: {info.get('status')}")
else:
    print("✓ No failed downloads.")

✓ No failed downloads.


In [72]:
# ── Minutes breakdown ─────────────────────────────────────────────────────
import pandas as pd

df   = pd.read_csv(PIPELINE_ROOT / "data/metadata.csv")
mins = df[df.doc_type == "minutes"]

print(f"Minutes total    : {len(mins)}")
print(f"Year range       : {mins.year.min()} – {mins.year.max()}")
print()
print("By chair regime:")
print(mins.groupby("chair_regime")["doc_id"].count().to_string())
print()
print("By year (last 10):")
print(mins.groupby("year")["doc_id"].count().tail(10).to_string())

Minutes total    : 183
Year range       : 2000 – 2026

By chair regime:
chair_regime
Bernanke     56
Greenspan    49
Powell       62
Yellen       16

By year (last 10):
year
2017    4
2018    4
2019    8
2020    8
2021    8
2022    8
2023    8
2024    8
2025    8
2026    2
